# Compile one intent for two connectivities

Goal: flip a target when four control qubits are all 1. The controls are prepared uniformly, so the output also checks coherent action. Keep the logical program unchanged while compiling for all-to-all and line connectivity. The compiler owns decomposition, routing and temporary wires.

In [ ]:
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QArray, QBit, X, allocate, control, hadamard_transform, qfunc

@qfunc
def conditional_flip(controls: Output[QArray[QBit]], target: Output[QBit]) -> None:
    allocate(4, controls)
    allocate(target)
    hadamard_transform(controls)
    control(controls, lambda: X(target))


In [ ]:
programs = {}
outcomes = {}
for topology in (None, "line"):
    label = "all-to-all" if topology is None else topology
    program = synth.synthesize(conditional_flip, topology=topology, basis=("cx", "u"), objective="depth", max_qubits=6, max_workspace=1, memory_limit_bytes=1024**2)
    programs[label] = program
    result = fq.run(program)
    outcomes[label] = dict(program.probabilities(result, registers=("target",)))
    print(label, outcomes[label], program.report["resources"])


## Read the trade-off

Both programs should give target=1 with probability 1/16. Their resources can differ because routing must respect the requested connectivity. `objective="depth"` requests an optimization objective; it does not certify a global optimum. Counts are compiler outputs for this version, not measured hardware runtimes.

Try lowering the width or workspace limit. An impossible request should be rejected explicitly. Physical placement and swaps remain compiler responsibilities.